# Market regimes, portfolio accounting and reproducible evidence

## 1. Research question and evidence tracks

Does a learned downside-risk score improve net risk-adjusted performance over a simple invested portfolio after accounting for exposure, distributions, cash, trading costs and the search over strategies?

This notebook is the explanatory interface to the tested `regime_research` package. Its default offline replay independently verifies the corrected historical research archive. An explicit `fit` run rebuilds the fixed HGB downside recipe on the identified SPY snapshot. Both use inspected history. Fresh forward evidence begins only after the active protocol's actual freeze and contemporaneously recorded decisions.

The archived historical study uses a non-investable S&P 500 price-index proxy. Its results omit distributions and assume zero cash return. The investable SPY track uses raw executable prices, action events and a lagged DFF-linked cash proxy. Payment dates are approximated by ex-date credit where unavailable. All inputs are immutable identified snapshots.

The original static KNN strategies and full-sample K-means analysis are preserved in the baseline archive as legacy and descriptive analyses.

In [ ]:
import json
import os
from pathlib import Path
import uuid

import pandas as pd
from IPython.display import Image, display
from regime_research.config import RunConfig
from regime_research.research import run_research, verify_study

project = Path(os.environ.get("REGIME_PROJECT", Path.cwd())).resolve()
configuration_path = os.environ.get("REGIME_RUN_CONFIG")
config = RunConfig.from_json(configuration_path) if configuration_path else RunConfig()
run_output = Path(os.environ.get("REGIME_RUN_OUTPUT", project / "runs" / ("interactive-" + uuid.uuid4().hex)))
display(pd.DataFrame([config.to_dict()]))

## 2. Decisions, labels and availability

A feature row is available one minute after its exchange close. Its decision executes at the following exchange open. The ordinary holding return ends at the next open after execution. Training and calibration use only fully matured labels.

For horizon $h$, the raw executable-price target return is

$$R_{t,h}=\frac{O_{t+h+1}}{O_{t+1}}-1.$$

The downside label is $y_t=1\{R_{t,h}<-b_{t,h}\}$ with

$$b_{t,h}=2k+0.5\,\widehat\sigma_t\sqrt h,$$

where $k$ is one-way cost and $\widehat\sigma_t$ is the trailing 20-session standard deviation of close returns. Label end is the exit-open timestamp. Base labels end strictly before calibration starts. Calibration labels end strictly before the refit. Inner selection returns end strictly before the selection decision.

The worked dates below illustrate the daily execution timeline.

In [ ]:
timeline = pd.DataFrame([
    {"Event": "Feature availability and decision", "UTC": "2020-01-02 21:01", "Information": "January 2 completed bar"},
    {"Event": "Execution", "UTC": "2020-01-03 14:30", "Information": "January 3 open fill"},
    {"Event": "Daily return end", "UTC": "2020-01-06 14:30", "Information": "January 6 open valuation"},
])
display(timeline)

## 3. Features, objectives and calibration

The causal feature builder preserves the corrected notebook's 98 indicators. They include relative moving averages, returns, volatility, volume changes, RSI, MACD, directional movement, normalized ranges and trend slopes. Their rolling calculations use information available through each row. A base-fit median imputer precedes supervised tree selection of 16 features. Every refit learns these transformations again from its own base block.

The fixed binary HGB model estimates $p_t=P(y_t=1\mid x_t)$ using class-balanced logistic loss

$$L(F)=\sum_i w_i\big[\log(1+e^{F(x_i)})-y_iF(x_i)\big],\qquad p_i=\frac{1}{1+e^{-F(x_i)}}.$$

At boosting iteration $m$, gradients and Hessians are $g_i=w_i(p_i-y_i)$ and $h_i=w_ip_i(1-p_i)$. A tree leaf's Newton update has the form

$$v_\ell=-\frac{\sum_{i\in\ell}g_i}{\sum_{i\in\ell}h_i+\lambda},\qquad F_m=F_{m-1}+\eta v_{\ell(x)}.$$

The registered implementation uses learning rate 0.05, 75 iterations, at most 15 leaves, minimum leaf size 25 and L2 value 1. Sigmoid calibration fits a separate probability map on a later purged block while freezing the base estimator. A missing required calibration class invokes the declared raw-probability fallback.

The recipe API also preserves the registered RF and return-regression families. RF averages class probabilities across trees. HGB regressors minimize squared error for return and risk targets. These families remain part of the disclosed historical search.

For allocation, scores are smoothed causally and ranked against a trailing 1,260-session history. The fixed rule targets full, half or zero exposure at the registered percentile thresholds. Immediate de-risking, a five-session hold and hysteretic re-entry define the state machine. A minimum 252 scored observations precedes evaluation.

In [ ]:
def show_progress(record):
    line = json.dumps(record, sort_keys=True)
    with (run_output / "refit_progress.jsonl").open("a", encoding="utf-8") as progress_file:
        progress_file.write(line + "\n")
        progress_file.flush()
    print(line)

study = run_research(project, run_output / "study", config, progress=show_progress)
display(pd.DataFrame([study.summary]))

## 4. Self-financing execution and a worked ledger

Portfolio wealth is $W=qP+C+A$, comprising shares, settled cash and distribution receivables. An order solves its post-cost target weight using actual traded notional. With equity value $E$, target $w$ and one-way cost $k$, a purchase satisfies

$$x=\frac{wW-E}{1+wk},$$

with the sale denominator $1-wk$. The ledger also respects settled-cash funding. Holdings drift between target changes. A distribution creates entitlement and later cash payment under the declared convention. Cash accrues using available rate observations and local-calendar ACT/360 intervals. Distribution cash remains in the cash account until the next rule target change. The buy-and-hold control retains its initial shares under this same policy, so its realized exposure drifts below one as cash distributions accumulate.

The example below is illustrative. It starts with $100 of wealth, reduces a $60 equity position to a 50% target, then applies a 5% price rise, one cash-accrual interval and a distribution entitlement. The P&L bridge separately tracks each component.

In [ ]:
from regime_research.execution import solve_target_trade

trade, cost, realized_weight, shortfall = solve_target_trade(60.0, 40.0, 0.0, 0.5, 0.0005)
equity_after_trade = 60.0 + trade
cash_after_trade = 40.0 - trade - cost
price_pnl = equity_after_trade * 0.05
cash_interest = cash_after_trade * 0.0001
distribution_entitlement = (equity_after_trade / 100.0) * 0.50
final_wealth = equity_after_trade + price_pnl + cash_after_trade + cash_interest + distribution_entitlement
ledger_example = pd.DataFrame([
    {"Component": "Initial capital", "Dollars": 100.0},
    {"Component": "Trading cost", "Dollars": -cost},
    {"Component": "Price P&L", "Dollars": price_pnl},
    {"Component": "Cash interest", "Dollars": cash_interest},
    {"Component": "Distribution entitlement", "Dollars": distribution_entitlement},
    {"Component": "Final wealth", "Dollars": final_wealth},
])
assert abs(final_wealth - (100.0 - cost + price_pnl + cash_interest + distribution_entitlement)) < 1e-12
display(ledger_example)

## 5. Frozen validation and research history

The active design is `downside-overlay` version `2.0.0`. It retains the fixed annual HGB downside overlay on SPY and buy-and-hold as the primary control. QQQ and IEF have predetermined 50/50 transfer weights. The minimum economic delta-Sharpe is 0.10. The primary interval has 95% confidence. Forward review requires 756 scheduled completed intervals and at least 10 invested episodes. Numerical risk, cost and inference thresholds are in the machine-readable protocol.

Version 2 binds the extracted package, exact runtime lock and strict runner. Its explicit UTC refits can use labels that have matured before the after-close refit timestamp. The archived date-bound implementation retains its own results. A new source version starts a new forward window at its actual freeze.

The append-only register preserves the original 182 candidates, their parent study, repeated attempts, failures, corrections and known diagnostics. Earlier informal search history is partially reconstructable. The historical max-T matrix includes the reported selector and has 183 streams. The single fixed fresh-evidence family and the broader research search have distinct counts.

Research validity requires chronology, accounting, provenance, reproducibility, trial disclosure and calibrated inference. Strategy evidence requires incremental net performance, uncertainty, risk, costs and adequate evaluation. Undefined or inadequate evidence is inconclusive. A valid negative result remains reportable.

In [ ]:
protocol = json.loads((project / config.protocol).read_text())
display(pd.DataFrame([{
    "Protocol": protocol["Protocol_ID"], "Version": protocol["Version"],
    "Primary comparator": protocol["Inference"]["Primary_Comparator"],
    "Evidence stage": study.summary["Stage"], "Forward observations": study.summary["Forward_Observations"],
    "Forward review intervals": protocol["Evaluation"]["Forward"]["Minimum_Completed_Intervals"],
}]))

## 6. Prediction, exposure and portfolio performance

Classification discrimination and calibrated probabilities are evaluated separately from portfolio P&L. A high ROC-AUC need not imply an economically valuable allocation after costs. The figures save their underlying table data alongside the PNG files. Legends and captions identify the instrument, evidence stage, cost and cash conventions.

The historical replay includes matched exposure and volatility controls, overlay ablations, randomized references and an independent P&L bridge. These comparisons test whether apparent timing benefits come from reduced market participation or generic trend/risk management.

In [ ]:
display(study.predictive)
display(study.metrics)
display(Image(filename=str(study.figures["wealth_and_exposure"])))

## 7. Dependence-aware uncertainty and the search over strategies

For net excess return $r_t-r_{f,t}$, annualized Sharpe is

$$\widehat S=\sqrt{252}\,\frac{\overline{r-r_f}}{s(r-r_f)}.$$

Delta-Sharpe compares the model with its registered control on identical dates. Paired circular-block resampling preserves their joint shocks and local serial dependence. HAC standard errors account for serial covariance. The joint max-T bootstrap adjusts the registered mean-difference tests across the supplied family.

Deflated Sharpe combines sample moments with the expected maximum Sharpe from the trial family. Its raw-count and dependence-sensitivity inputs remain visible. Its analytical selected-return approximation assumes IID sampling. The dependence-aware paired uncertainty and calibrated mean test are separate checks. Repeatedly inspected history remains retrospective even when within-period refits and selections are causal.

In [ ]:
display(study.inference)

## 8. Robustness, limitations and interpretation

The corrected historical nested strategy has net Sharpe about 0.452. SMA 50/200, past exposure/volatility controls and the invested core have stronger point estimates. The model increment versus the core is negative and its paired interval includes zero. The evidence supports a careful research result with limited demonstrated model increment.

A package `fit` run evaluates a distinct explicit-timestamp implementation on already inspected SPY history. Its saved results are developmental. It also reports the registered 0, 5, 10 and 20 bps cost grid. Newly favorable point estimates on inspected history do not satisfy the fresh forward requirement.

Remaining empirical limitations include revised historical vintages, the dividend-payment approximation, the DFF cash proxy, omitted taxes and market impact, correlated transfer markets and incomplete earlier informal search history. Transfer and forward evaluations await their registered inputs and contemporaneously recorded signals. No forward outcome is inferred from this replay.

In [ ]:
display(pd.DataFrame([verify_study(study.directory)]))
print(study.summary["Interpretation"])

## 9. Reproduce and inspect the receipts

Install the exact Windows CPython 3.13.7 environment from `requirements.lock`, then install the local package. The release archive identified by `archives/layer5_baseline/manifest.json` supplies the immutable historical data and results.

Run the default offline verification and notebook replay with

```powershell
python -m regime_research.runner run --mode replay --output runs/release_replay
python -m regime_research.runner verify runs/release_replay
```

An explicit development refit uses `--mode fit`. It uses the selected immutable snapshot. `python data.py refresh-spy` is the separate explicit network-acquisition command. Complete per-recipe/refit checkpoints are validated before reuse. Failed or interrupted work retains its progress and failure records.

The supported runner creates a private fresh kernel, enforces every saved code-cell ID and source in exact order, records request IDs and kernel execution counts, checks source stability, verifies accounting and key metrics independently, then publishes the completed-run manifest. Extra, reordered, nested or failed execution invalidates the receipt. Scientific source, protocol, data, runtime lock, executed notebook, outputs, Git revision and anchored experiment history are bound in that manifest.

[Reproduction guide](docs/REPRODUCIBILITY.md) · [Layer specification](QUANT_RESEARCH_SPEC.md) · [Validation protocol](protocols/layer6_validation_v2.json) · [Historical baseline manifest](archives/layer5_baseline/manifest.json)